# LEBER A1 - Symmetric Bilateral Features - ResNet50 - BCE - Seed 42

A1 menguji fitur bilateral simetris berupa jumlah, selisih absolut, dan perkalian fitur kedua mata. Backbone, loss, split, dan konfigurasi training sama dengan A0.

In [ ]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path(
    "/kaggle/input/datasets/kevinardhana/"
    "odir-5k-patient-level-multi-label-fundus-dataset"
)

IMAGE_DIR = DATASET_DIR / "Training Images" / "Training Images"

TRAIN_CSV = DATASET_DIR / "train.csv"
VALID_CSV = DATASET_DIR / "validation.csv"

LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

print("Train      :", train_df.shape)
print("Validation :", valid_df.shape)
print("\nKolom:")
print(train_df.columns.tolist())

assert len(train_df) == 2450
assert len(valid_df) == 525
assert all(column in train_df.columns for column in LABELS)

print("\nContoh data:")
display(train_df.head())

print("\nCitra kiri contoh ada :", (IMAGE_DIR / train_df.loc[0, "left_image"]).is_file())
print("Citra kanan contoh ada:", (IMAGE_DIR / train_df.loc[0, "right_image"]).is_file())

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

for path in INPUT_ROOT.rglob("train.csv"):
    print("train.csv ditemukan di:", path)

for path in INPUT_ROOT.rglob("0_left.jpg"):
    print("Contoh citra ditemukan di:", path)

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


class FundusPairDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        left_path = self.image_dir / row["left_image"]
        right_path = self.image_dir / row["right_image"]

        left_image = Image.open(left_path).convert("RGB")
        right_image = Image.open(right_path).convert("RGB")

        left_image = self.transform(left_image)
        right_image = self.transform(right_image)

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": labels,
            "patient_id": int(row["patient_id"]),
        }


train_dataset = FundusPairDataset(train_df, IMAGE_DIR, train_transform)
valid_dataset = FundusPairDataset(valid_df, IMAGE_DIR, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


batch = next(iter(train_loader))

print("Citra kiri :", batch["left_image"].shape)
print("Citra kanan:", batch["right_image"].shape)
print("Label      :", batch["labels"].shape)
print("Contoh target:", batch["labels"][0].tolist())

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


class SymmetricBilateralResNet50(nn.Module):
    def __init__(self, num_labels=8, dropout=0.30, pretrained=True):
        super().__init__()

        weights = (
            ResNet50_Weights.IMAGENET1K_V2
            if pretrained else None
        )
        self.backbone = resnet50(weights=weights)

        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim * 3, num_labels)
        )

    @staticmethod
    def symmetric_features(left_features, right_features):
        sum_features = left_features + right_features
        difference_features = torch.abs(left_features - right_features)
        product_features = left_features * right_features
        return torch.cat(
            [sum_features, difference_features, product_features],
            dim=1
        )

    def forward(self, left_image, right_image):
        left_features = self.backbone(left_image)
        right_features = self.backbone(right_image)
        bilateral_features = self.symmetric_features(
            left_features,
            right_features
        )
        return self.classifier(bilateral_features)


model = SymmetricBilateralResNet50(
    num_labels=len(LABELS)
).to(DEVICE)

left_batch = batch["left_image"].to(DEVICE)
right_batch = batch["right_image"].to(DEVICE)

model.eval()
with torch.no_grad():
    logits = model(left_batch, right_batch)
    swapped_logits = model(right_batch, left_batch)
    probabilities = torch.sigmoid(logits)

swap_error = torch.max(
    torch.abs(logits - swapped_logits)
).item()

assert logits.shape == (BATCH_SIZE, len(LABELS))
assert swap_error < 1e-6, (
    f"Fitur A1 tidak invariant terhadap swap: {swap_error}"
)

print("Logits shape       :", logits.shape)
print("Probabilities shape:", probabilities.shape)
print("Maksimum selisih logit setelah swap:", swap_error)
print(
    "Parameter trainable:",
    sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
)

In [ ]:
import json
from pathlib import Path
from sklearn.metrics import f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

OUTPUT_DIR = Path("/kaggle/working/leber_a1_symmetric_bce_512_seed42")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=device.type == "cuda"
        ):
            logits = model(left_images, right_images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=device.type == "cuda"
        ):
            logits = model(left_images, right_images)
            loss = criterion(logits, labels)
        probabilities = torch.sigmoid(logits.float())

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_targets.append(labels.cpu().numpy())
        all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probabilities)
    y_pred = (y_prob >= threshold).astype(int)

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "loss": total_loss / total_samples,
        "macro_f1": macro_f1,
        "y_true": y_true,
        "y_prob": y_prob,
    }


print("BCE, AdamW, scheduler, dan fungsi training siap.")

In [ ]:
history = []
best_macro_f1 = -1.0
best_epoch = 0
patience_counter = 0

checkpoint_path = OUTPUT_DIR / "best_leber_a1_symmetric_resnet50_bce_512_seed42.pt"
history_path = OUTPUT_DIR / "training_history.json"

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE
    )

    validation_result = evaluate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE,
        threshold=0.5
    )

    validation_loss = validation_result["loss"]
    validation_macro_f1 = validation_result["macro_f1"]

    scheduler.step(validation_macro_f1)

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_result = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_loss": float(validation_loss),
        "validation_macro_f1": float(validation_macro_f1),
        "learning_rate": float(current_lr),
    }
    history.append(epoch_result)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {validation_loss:.4f} | "
        f"val Macro-F1: {validation_macro_f1:.4f} | "
        f"lr: {current_lr:.6f}"
    )

    if validation_macro_f1 > best_macro_f1:
        best_macro_f1 = validation_macro_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_macro_f1": best_macro_f1,
                "labels": LABELS,
                "threshold": 0.5,
                "backbone": "shared_resnet50_symmetric_features",
                "input_size": 512,
                "batch_size": 16,
                "seed": 42,
                "loss_function": "BCEWithLogitsLoss",
            },
            checkpoint_path
        )

        print("  ✓ Checkpoint terbaik disimpan.")

    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping pada epoch {epoch}. "
            f"Checkpoint terbaik: epoch {best_epoch}."
        )
        break


with history_path.open("w") as file:
    json.dump(history, file, indent=2)

print("\nPelatihan selesai.")
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_macro_f1, 4))
print("Checkpoint:", checkpoint_path)

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/leber_a1_symmetric_bce_512_seed42")
checkpoint_path = OUTPUT_DIR / "best_leber_a1_symmetric_resnet50_bce_512_seed42.pt"

print("Lokasi checkpoint:", checkpoint_path)
print("Checkpoint tersedia:", checkpoint_path.exists())

In [ ]:
# Memuat checkpoint terbaik dan menentukan threshold BCE
# menggunakan validation set

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50


LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]

BATCH_SIZE = 16
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Lokasi dataset dan hasil training
DATASET_DIR = Path(
    "/kaggle/input/datasets/kevinardhana/"
    "odir-5k-patient-level-multi-label-fundus-dataset"
)

IMAGE_DIR = (
    DATASET_DIR
    / "Training Images"
    / "Training Images"
)

VALID_CSV = DATASET_DIR / "validation.csv"

OUTPUT_DIR = Path(
    "/kaggle/working/leber_a1_symmetric_bce_512_seed42"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Dataset pasangan citra mata kiri dan kanan
class FundusPairDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        transform
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        left_path = (
            self.image_dir
            / row["left_image"]
        )

        right_path = (
            self.image_dir
            / row["right_image"]
        )

        left_image = Image.open(
            left_path
        ).convert("RGB")

        right_image = Image.open(
            right_path
        ).convert("RGB")

        left_image = self.transform(
            left_image
        )

        right_image = self.transform(
            right_image
        )

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": labels
        }


# Arsitektur harus sama dengan model saat training
class SymmetricBilateralResNet50(nn.Module):
    def __init__(self, num_labels=8, dropout=0.30):
        super().__init__()
        self.backbone = resnet50(weights=None)
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim * 3, num_labels)
        )

    @staticmethod
    def symmetric_features(left_features, right_features):
        return torch.cat(
            [
                left_features + right_features,
                torch.abs(left_features - right_features),
                left_features * right_features
            ],
            dim=1
        )

    def forward(self, left_image, right_image):
        left_features = self.backbone(left_image)
        right_features = self.backbone(right_image)
        bilateral_features = self.symmetric_features(
            left_features,
            right_features
        )
        return self.classifier(bilateral_features)


@torch.no_grad()
def collect_predictions(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(
            device,
            non_blocking=True
        )

        right_images = batch["right_image"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        logits = model(
            left_images,
            right_images
        )

        probabilities = torch.sigmoid(
            logits
        )

        all_targets.append(
            labels.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    targets = np.concatenate(
        all_targets,
        axis=0
    )

    probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    return targets, probabilities


# Transformasi validation sama dengan baseline
eval_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Membaca validation set
valid_df = pd.read_csv(
    VALID_CSV
)

assert len(valid_df) == 525, (
    f"Jumlah validation tidak sesuai: {len(valid_df)}"
)

assert all(
    label in valid_df.columns
    for label in LABELS
), "Kolom label validation tidak lengkap"


valid_dataset = FundusPairDataset(
    dataframe=valid_df,
    image_dir=IMAGE_DIR,
    transform=eval_transform
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)


# Checkpoint dibuat oleh cell training sebelumnya
checkpoint_path = (
    OUTPUT_DIR
    / "best_leber_a1_symmetric_resnet50_bce_512_seed42.pt"
)

assert checkpoint_path.is_file(), (
    "Checkpoint tidak ditemukan. "
    f"Lokasi yang diperiksa: {checkpoint_path}"
)


checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)


required_checkpoint_keys = {
    "epoch",
    "model_state_dict"
}

missing_keys = (
    required_checkpoint_keys
    - set(checkpoint.keys())
)

assert not missing_keys, (
    "Isi checkpoint tidak lengkap. "
    f"Key yang tidak ditemukan: {missing_keys}"
)


# Membuat model dan memuat bobot terbaik
model = SymmetricBilateralResNet50(
    num_labels=len(LABELS),
    dropout=0.30
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


print("Checkpoint ditemukan:", checkpoint_path)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Device evaluasi:", DEVICE)


# Menghasilkan probabilitas validation
y_val, prob_val = collect_predictions(
    model=model,
    loader=valid_loader,
    device=DEVICE
)


assert y_val.shape == (525, 8), (
    f"Bentuk target validation tidak sesuai: {y_val.shape}"
)

assert prob_val.shape == (525, 8), (
    "Bentuk probabilitas validation "
    f"tidak sesuai: {prob_val.shape}"
)


# Mencari threshold terbaik untuk setiap label
candidate_thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

best_thresholds = []
threshold_rows = []


for label_index, label_name in enumerate(LABELS):
    label_scores = []

    for threshold in candidate_thresholds:
        label_predictions = (
            prob_val[:, label_index]
            >= threshold
        ).astype(np.int32)

        score = f1_score(
            y_val[:, label_index],
            label_predictions,
            zero_division=0
        )

        label_scores.append(score)

    best_position = int(
        np.argmax(label_scores)
    )

    selected_threshold = float(
        candidate_thresholds[best_position]
    )

    selected_f1 = float(
        label_scores[best_position]
    )

    best_thresholds.append(
        selected_threshold
    )

    threshold_rows.append({
        "label": label_name,
        "threshold_terbaik":
            selected_threshold,
        "F1_validation":
            selected_f1,
        "jumlah_positif_validation":
            int(y_val[:, label_index].sum())
    })


best_thresholds = np.asarray(
    best_thresholds,
    dtype=np.float32
)


# Membandingkan threshold 0,50 dengan threshold per label
default_predictions = (
    prob_val >= 0.50
).astype(np.int32)

optimized_predictions = (
    prob_val >= best_thresholds
).astype(np.int32)


macro_f1_default = f1_score(
    y_val,
    default_predictions,
    average="macro",
    zero_division=0
)

macro_f1_optimized = f1_score(
    y_val,
    optimized_predictions,
    average="macro",
    zero_division=0
)


threshold_df = pd.DataFrame(
    threshold_rows
)


print(
    "\nMacro-F1 threshold 0,50:",
    f"{macro_f1_default:.4f}"
)

print(
    "Macro-F1 threshold per label:",
    f"{macro_f1_optimized:.4f}"
)

print("\nThreshold setiap label:")
display(threshold_df)


# Menyimpan threshold
threshold_csv_path = (
    OUTPUT_DIR
    / "validation_thresholds_leber_a1_symmetric_bce_512_seed42.csv"
)

threshold_json_path = (
    OUTPUT_DIR
    / "validation_thresholds_leber_a1_symmetric_bce_512_seed42.json"
)


threshold_df.to_csv(
    threshold_csv_path,
    index=False
)


with open(
    threshold_json_path,
    "w"
) as file:
    json.dump(
        {
            "checkpoint_epoch":
                int(checkpoint["epoch"]),
            "labels":
                LABELS,
            "thresholds":
                best_thresholds.tolist(),
            "macro_f1_threshold_0_5":
                float(macro_f1_default),
            "macro_f1_optimized":
                float(macro_f1_optimized)
        },
        file,
        indent=2
    )


print("\nThreshold berhasil disimpan:")
print("-", threshold_csv_path)
print("-", threshold_json_path)

In [ ]:
# Audit konsistensi pertukaran mata pada validation set

@torch.no_grad()
def collect_original_and_swapped_probabilities(model, loader, device):
    model.eval()
    all_targets = []
    original_probabilities = []
    swapped_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].cpu().numpy()

        original_logits = model(left_images, right_images)
        swapped_logits = model(right_images, left_images)

        all_targets.append(labels)
        original_probabilities.append(
            torch.sigmoid(original_logits).cpu().numpy()
        )
        swapped_probabilities.append(
            torch.sigmoid(swapped_logits).cpu().numpy()
        )

    return (
        np.concatenate(all_targets),
        np.concatenate(original_probabilities),
        np.concatenate(swapped_probabilities)
    )


y_swap, probabilities_original, probabilities_swapped = (
    collect_original_and_swapped_probabilities(
        model=model,
        loader=valid_loader,
        device=DEVICE
    )
)

absolute_probability_difference = np.abs(
    probabilities_original - probabilities_swapped
)

predictions_original = (
    probabilities_original >= best_thresholds
).astype(np.int32)

predictions_swapped = (
    probabilities_swapped >= best_thresholds
).astype(np.int32)

swap_metrics = {
    "split": "validation",
    "mean_absolute_probability_difference": float(
        absolute_probability_difference.mean()
    ),
    "maximum_absolute_probability_difference": float(
        absolute_probability_difference.max()
    ),
    "label_decision_disagreement_rate": float(
        np.not_equal(
            predictions_original,
            predictions_swapped
        ).mean()
    ),
    "patient_exact_agreement_rate": float(
        np.all(
            predictions_original == predictions_swapped,
            axis=1
        ).mean()
    )
}

assert swap_metrics["mean_absolute_probability_difference"] < 1e-6
assert swap_metrics["label_decision_disagreement_rate"] == 0.0

swap_metrics_path = (
    OUTPUT_DIR
    / "validation_swap_metrics_leber_a1_symmetric_bce_512_seed42.json"
)

with open(swap_metrics_path, "w") as file:
    json.dump(swap_metrics, file, indent=2)

print("Audit swap validation A1:")
for key, value in swap_metrics.items():
    print(f"{key}: {value}")

print("\nArtefak disimpan di:", OUTPUT_DIR)
print("Test set belum digunakan pada tahap pemilihan arsitektur A1.")